# MedSAM2 for Brain Ventricle Segmentation

**Author:** Matheus Machado Rech

---

This notebook adapts [MedSAM2](https://github.com/bowang-lab/MedSAM2) for automated ventricle segmentation from head CT NIfTI volumes. MedSAM2 treats 3D medical volumes as "video" sequences, where each axial slice is a frame.

**Pipeline:**
1. Load a brain CT NIfTI volume
2. Apply brain windowing (WL=40, WW=80) and normalize to [0, 255]
3. Find the axial slice with maximum ventricle area using classical HU thresholding
4. Prompt MedSAM2 with a bounding box on that key slice
5. Propagate segmentation forward (superior) and backward (inferior) through the volume
6. Post-process with largest connected component extraction
7. Compute morphometric indices: Evans Index, Ventricle Volume, NPH Score

**Use case:** Normal Pressure Hydrocephalus (NPH) morphometric analysis.

**Requirements:** Google Colab with T4 GPU runtime.

**License:** Research use only -- not for clinical diagnosis.

---

### Table of Contents
1. [Environment Setup](#environment-setup)
2. [Download Sample Brain CT Data](#download-sample-data)
3. [Load and Preprocess CT Volume](#load-preprocess)
4. [Find Key Ventricle Slice](#find-key-slice)
5. [Run MedSAM2 Inference](#medsam2-inference)
6. [Visualize Results](#visualize-results)
7. [Compute Morphometric Indices](#morphometrics)
8. [Save Results](#save-results)
9. [Next Steps](#next-steps)

In [ ]:
# =============================================================================
# 1. Environment Setup
# =============================================================================
# Clone MedSAM2 repository and install dependencies
# Estimated time: ~3-5 minutes on Colab

import os
import sys
import hashlib

# Verify GPU availability
import subprocess
gpu_info = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                          capture_output=True, text=True)
print(f"GPU: {gpu_info.stdout.strip()}")
assert 'T4' in gpu_info.stdout or 'GPU' in gpu_info.stdout, \
    "No GPU detected. Go to Runtime > Change runtime type > T4 GPU"

# Clone MedSAM2 at a pinned commit
MEDSAM2_COMMIT = '332f30d420f1d1b08e2a79b3ae6a602458808383'
if not os.path.exists('/content/MedSAM2'):
    !git clone https://github.com/bowang-lab/MedSAM2.git /content/MedSAM2
    print("MedSAM2 cloned successfully")
else:
    print("MedSAM2 already cloned")

%cd /content/MedSAM2
!git fetch --depth 1 origin 332f30d420f1d1b08e2a79b3ae6a602458808383
!git checkout 332f30d420f1d1b08e2a79b3ae6a602458808383

# Install MedSAM2 in editable mode
!pip install -e . -q

# Install additional dependencies
!pip install nibabel scikit-image scipy -q

# Download MedSAM2 checkpoint (~149 MB) from a pinned revision
MEDSAM2_HF_REVISION = 'e4a6f35edd7e091619cbc0750f462f1574e23955'
EXPECTED_CHECKPOINT_SHA256 = 'dcd946a4d934f553236866fc7e8af77f7e931430e9c044f4ac9d6a723630a870'

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

os.makedirs('checkpoints', exist_ok=True)
checkpoint_path = 'checkpoints/MedSAM2_2411.pt'
if not os.path.exists(checkpoint_path):
    !wget -q --show-progress -P checkpoints \
        https://huggingface.co/wanglab/MedSAM2/resolve/e4a6f35edd7e091619cbc0750f462f1574e23955/MedSAM2_2411.pt
    print(f"Checkpoint downloaded: {os.path.getsize(checkpoint_path) / 1e6:.1f} MB")
else:
    print(f"Checkpoint already exists: {os.path.getsize(checkpoint_path) / 1e6:.1f} MB")

actual_sha256 = sha256_file(checkpoint_path)
if actual_sha256 != EXPECTED_CHECKPOINT_SHA256:
    raise RuntimeError(
        f"Checkpoint SHA256 mismatch: {actual_sha256} != {EXPECTED_CHECKPOINT_SHA256}"
    )
print(f"Checkpoint SHA256 verified: {actual_sha256}")

# Verify imports
import torch
import numpy as np
import nibabel as nib
from PIL import Image
import matplotlib.pyplot as plt
from skimage import measure
from scipy.ndimage import binary_fill_holes, binary_erosion

print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"\nEnvironment ready.")

<a id="download-sample-data"></a>
## 2. Download Sample Brain CT Data

For testing the pipeline, we generate a **synthetic brain CT volume** with realistic geometry:
- Ellipsoidal brain parenchyma (HU ~30)
- Skull shell (HU ~700)
- Bilateral lateral ventricles filled with CSF (HU ~10)
- Volume dimensions: 256 x 256 x 128 with 1.0 x 1.0 x 1.5 mm spacing

If you have a real brain CT NIfTI file, upload it to Colab and set `nifti_path` to its location. The pipeline expects standard head CT Hounsfield units.

In [ ]:
# =============================================================================
# 2. Download / Generate Sample Brain CT Data
# =============================================================================
# Option A: Upload your own NIfTI file
# from google.colab import files
# uploaded = files.upload()
# nifti_path = list(uploaded.keys())[0]

# Option B: Generate synthetic brain CT for pipeline testing
import numpy as np
import nibabel as nib

os.makedirs('data', exist_ok=True)

def create_synthetic_brain_ct(output_path='data/synthetic_brain_ct.nii.gz'):
    """
    Create a synthetic brain CT volume with ventricles for testing.

    Generates a 256x256x128 volume with:
    - Ellipsoidal skull shell (HU ~700)
    - Brain parenchyma (HU ~30)
    - Bilateral lateral ventricles with CSF (HU ~10)
    - Realistic voxel spacing (1.0 x 1.0 x 1.5 mm)
    """
    shape = (256, 256, 128)
    volume = np.full(shape, -1000, dtype=np.int16)  # Air background

    cx, cy = shape[0] // 2, shape[1] // 2
    cz = shape[2] // 2

    # Vectorized coordinate grids for speed
    x = np.arange(shape[0])
    y = np.arange(shape[1])
    z = np.arange(shape[2])
    X, Y, Z = np.meshgrid(x, y, z, indexing='ij')

    # Normalized distances for ellipsoidal shapes
    dx_brain = (X - cx) / 90.0
    dy_brain = (Y - cy) / 100.0
    dz_brain = (Z - cz) / 44.0
    r_brain = dx_brain**2 + dy_brain**2 + dz_brain**2

    dx_skull = (X - cx) / 95.0
    dy_skull = (Y - cy) / 105.0
    dz_skull = (Z - cz) / 47.0
    r_skull = dx_skull**2 + dy_skull**2 + dz_skull**2

    # Skull shell (HU ~700)
    skull_mask = (r_skull <= 1.1) & (r_skull >= 0.92)
    volume[skull_mask] = 700

    # Brain parenchyma (HU ~30) inside the skull
    brain_mask = r_brain <= 1.0
    volume[brain_mask & ~skull_mask] = 30

    # Add some soft tissue variation
    rng = np.random.default_rng(42)
    noise = rng.normal(0, 5, shape).astype(np.int16)
    volume[brain_mask & ~skull_mask] += noise[brain_mask & ~skull_mask]

    # Left lateral ventricle (HU ~10, CSF)
    dx_lv = (X - (cx - 15)) / 13.0
    dy_lv = (Y - cy) / 22.0
    dz_lv = (Z - cz) / 19.0
    left_vent = (dx_lv**2 + dy_lv**2 + dz_lv**2) <= 1.0
    volume[left_vent] = 10

    # Right lateral ventricle (HU ~10, CSF)
    dx_rv = (X - (cx + 15)) / 13.0
    dy_rv = (Y - cy) / 22.0
    dz_rv = (Z - cz) / 19.0
    right_vent = (dx_rv**2 + dy_rv**2 + dz_rv**2) <= 1.0
    volume[right_vent] = 10

    # Third ventricle (small, midline)
    dx_3v = (X - cx) / 3.0
    dy_3v = (Y - cy) / 8.0
    dz_3v = (Z - cz) / 10.0
    third_vent = (dx_3v**2 + dy_3v**2 + dz_3v**2) <= 1.0
    volume[third_vent] = 10

    # Create NIfTI
    spacing = (1.0, 1.0, 1.5)  # mm
    affine = np.diag([spacing[0], spacing[1], spacing[2], 1.0])
    img = nib.Nifti1Image(volume, affine)
    nib.save(img, output_path)

    # Report statistics
    vent_voxels = np.sum(left_vent | right_vent | third_vent)
    voxel_vol_mm3 = np.prod(spacing)
    vent_vol_ml = vent_voxels * voxel_vol_mm3 / 1000.0

    print(f"Synthetic brain CT saved: {output_path}")
    print(f"  Shape: {shape}")
    print(f"  Spacing: {spacing} mm")
    print(f"  File size: {os.path.getsize(output_path) / 1e6:.1f} MB")
    print(f"  Ventricle voxels: {vent_voxels}")
    print(f"  Estimated ventricle volume: {vent_vol_ml:.1f} mL")
    return output_path

nifti_path = create_synthetic_brain_ct()

# Preview mid-axial slice
nii_preview = nib.load(nifti_path)
vol_preview = nii_preview.get_fdata()
mid_z = vol_preview.shape[2] // 2

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(vol_preview[:, :, mid_z].T, cmap='gray', origin='lower')
axes[0].set_title(f'Axial (z={mid_z})')
axes[0].axis('off')

mid_y = vol_preview.shape[1] // 2
axes[1].imshow(vol_preview[:, mid_y, :].T, cmap='gray', origin='lower')
axes[1].set_title(f'Coronal (y={mid_y})')
axes[1].axis('off')

mid_x = vol_preview.shape[0] // 2
axes[2].imshow(vol_preview[mid_x, :, :].T, cmap='gray', origin='lower')
axes[2].set_title(f'Sagittal (x={mid_x})')
axes[2].axis('off')

plt.suptitle('Synthetic Brain CT Volume (raw HU)', fontsize=14)
plt.tight_layout()
plt.show()

<a id="load-preprocess"></a>
## 3. Load and Preprocess CT Volume

**Brain windowing** enhances soft tissue contrast:
- **Window Level (WL):** 40 HU -- center of the intensity range
- **Window Width (WW):** 80 HU -- range of intensities displayed

This maps HU [0, 80] to pixel values [0, 255], making ventricles (CSF ~5-15 HU) and brain parenchyma (~20-45 HU) clearly distinguishable.

**Preprocessing steps:**
1. Load NIfTI and extract voxel data + spacing
2. Reorient to RAS+ standard orientation
3. Apply brain window: clip HU to [0, 80], normalize to [0, 255]
4. Transpose to (Z, Y, X) = (slices, height, width) for MedSAM2's "video" format

In [ ]:
# =============================================================================
# 3. Load and Preprocess CT Volume
# =============================================================================
import numpy as np
import nibabel as nib
from PIL import Image
import matplotlib.pyplot as plt

# Load NIfTI
nii = nib.load(nifti_path)
volume = nii.get_fdata().astype(np.float32)
spacing = nii.header.get_zooms()[:3]
print(f"Volume shape (original): {volume.shape}")
print(f"Voxel spacing: {spacing} mm")
print(f"HU range: [{volume.min():.0f}, {volume.max():.0f}]")

# Reorient to RAS+ if needed
ornt = nib.orientations.io_orientation(nii.affine)
ras_ornt = nib.orientations.axcodes2ornt(('R', 'A', 'S'))
if not np.array_equal(ornt, ras_ornt):
    transform = nib.orientations.ornt_transform(ornt, ras_ornt)
    volume = nib.orientations.apply_orientation(volume, transform)
    print(f"Volume reoriented to RAS+, new shape: {volume.shape}")
else:
    print("Volume already in RAS+ orientation")

# Brain window: WL=40, WW=80 --> clip to [0, 80]
WL, WW = 40, 80
lower_bound = WL - WW // 2  # 0
upper_bound = WL + WW // 2  # 80
volume_windowed = np.clip(volume, lower_bound, upper_bound)
volume_uint8 = ((volume_windowed - lower_bound) / (upper_bound - lower_bound) * 255).astype(np.uint8)

print(f"\nBrain window applied: WL={WL}, WW={WW}")
print(f"Clipped HU range: [{lower_bound}, {upper_bound}]")
print(f"Output pixel range: [{volume_uint8.min()}, {volume_uint8.max()}]")

# For MedSAM2: axial slices are along axis 2 (I-S in RAS+)
# Transpose to (Z, Y, X) = (slices, height, width) for "video" format
img_3D = np.transpose(volume_uint8, (2, 1, 0))  # (Z, Y, X)
print(f"\nMedSAM2 input shape (Z, Y, X): {img_3D.shape}")
print(f"  Z (slices): {img_3D.shape[0]}")
print(f"  Y (height): {img_3D.shape[1]}")
print(f"  X (width):  {img_3D.shape[2]}")

# Visualize brain-windowed slices
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
n_slices = img_3D.shape[0]
for i, frac in enumerate([0.3, 0.4, 0.5, 0.6]):
    z = int(n_slices * frac)
    axes[i].imshow(img_3D[z], cmap='gray')
    axes[i].set_title(f'Slice {z}/{n_slices}')
    axes[i].axis('off')
plt.suptitle('Brain-Windowed Axial Slices', fontsize=14)
plt.tight_layout()
plt.show()

<a id="find-key-slice"></a>
## 4. Find Key Ventricle Slice (Classical Pipeline)

MedSAM2 requires a **prompt** on a single frame to initiate segmentation. We use classical HU thresholding to automatically find the best slice and bounding box:

1. **CSF thresholding:** Voxels with HU in [0, 22] are candidate CSF
2. **Central crop:** Restrict to the central 60% of each slice to exclude peripheral CSF (sulci, cisterns)
3. **Key slice selection:** The axial slice with the most central CSF voxels
4. **Bounding box:** Tight box around the CSF region on the key slice, with 10% margin

This matches the approach used in HydroMorph RN's classical pipeline (`src/pipeline/Morphometrics.js`).

In [ ]:
# =============================================================================
# 4. Find Key Ventricle Slice and Bounding Box
# =============================================================================

def find_ventricle_slice_and_bbox(nifti_path, lower_hu=0, upper_hu=22):
    """
    Find the axial slice with the most CSF/ventricle voxels
    and compute a bounding box around them.

    Uses simple HU thresholding: CSF is typically 0-22 HU in head CT.
    Restricts search to the central 60% of each slice to exclude
    peripheral CSF in sulci and cisterns.

    Parameters
    ----------
    nifti_path : str
        Path to the NIfTI file (original HU values).
    lower_hu : float
        Lower HU threshold for CSF (default: 0).
    upper_hu : float
        Upper HU threshold for CSF (default: 22).

    Returns
    -------
    key_slice_idx : int
        Index of the axial slice with maximum ventricle area.
    bbox : np.ndarray
        Bounding box [x1, y1, x2, y2] in pixel coordinates.
    """
    # Load original HU volume for thresholding
    nii_orig = nib.load(nifti_path)
    vol_hu = nii_orig.get_fdata().astype(np.float32)

    # Reorient to RAS+ to match preprocessed volume
    ornt = nib.orientations.io_orientation(nii_orig.affine)
    ras_ornt = nib.orientations.axcodes2ornt(('R', 'A', 'S'))
    if not np.array_equal(ornt, ras_ornt):
        transform = nib.orientations.ornt_transform(ornt, ras_ornt)
        vol_hu = nib.orientations.apply_orientation(vol_hu, transform)

    # Transpose to (Z, Y, X) to match MedSAM2 format
    vol_hu = np.transpose(vol_hu, (2, 1, 0))

    # CSF mask by HU thresholding
    csf_mask = (vol_hu >= lower_hu) & (vol_hu <= upper_hu)

    # Restrict to central 60% (exclude peripheral CSF)
    D, H, W = csf_mask.shape
    y_start, y_end = int(H * 0.2), int(H * 0.8)
    x_start, x_end = int(W * 0.2), int(W * 0.8)
    central_mask = np.zeros_like(csf_mask)
    central_mask[:, y_start:y_end, x_start:x_end] = \
        csf_mask[:, y_start:y_end, x_start:x_end]

    # Find slice with most ventricle voxels
    slice_counts = central_mask.sum(axis=(1, 2))
    key_slice = int(np.argmax(slice_counts))

    print(f"CSF voxel counts per slice (top 5):")
    top_slices = np.argsort(slice_counts)[::-1][:5]
    for s in top_slices:
        print(f"  Slice {s}: {int(slice_counts[s])} voxels")

    # Compute bounding box on key slice
    mask_2d = central_mask[key_slice]
    rows = np.any(mask_2d, axis=1)
    cols = np.any(mask_2d, axis=0)

    if not rows.any():
        print("WARNING: No CSF detected. Using center fallback.")
        return key_slice, np.array([W // 4, H // 4, 3 * W // 4, 3 * H // 4])

    rmin, rmax = np.where(rows)[0][[0, -1]]
    cmin, cmax = np.where(cols)[0][[0, -1]]

    # Add 10% margin for robustness
    margin_y = max(int((rmax - rmin) * 0.1), 5)
    margin_x = max(int((cmax - cmin) * 0.1), 5)
    bbox = np.array([
        max(0, cmin - margin_x),
        max(0, rmin - margin_y),
        min(W - 1, cmax + margin_x),
        min(H - 1, rmax + margin_y)
    ])

    return key_slice, bbox


key_slice_idx, ventricle_bbox = find_ventricle_slice_and_bbox(nifti_path)
print(f"\nKey ventricle slice: {key_slice_idx}")
print(f"Bounding box (x1, y1, x2, y2): {ventricle_bbox}")
print(f"Box size: {ventricle_bbox[2]-ventricle_bbox[0]} x {ventricle_bbox[3]-ventricle_bbox[1]} pixels")

# Visualize the key slice with bounding box prompt
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Original slice
axes[0].imshow(img_3D[key_slice_idx], cmap='gray')
axes[0].set_title(f'Key Slice #{key_slice_idx} (brain window)')
axes[0].axis('off')

# With bounding box
axes[1].imshow(img_3D[key_slice_idx], cmap='gray')
from matplotlib.patches import Rectangle
x1, y1, x2, y2 = ventricle_bbox
rect = Rectangle((x1, y1), x2 - x1, y2 - y1,
                  linewidth=2, edgecolor='cyan', facecolor='none',
                  linestyle='--')
axes[1].add_patch(rect)
axes[1].set_title('Bounding Box Prompt for MedSAM2')
axes[1].axis('off')

# Neighboring slices context
n_context = 5
context_slices = range(max(0, key_slice_idx - n_context),
                       min(img_3D.shape[0], key_slice_idx + n_context + 1))
montage = np.hstack([img_3D[z] for z in context_slices])
axes[2].imshow(montage, cmap='gray')
axes[2].set_title(f'Context: slices {min(context_slices)}-{max(context_slices)}')
axes[2].axis('off')

plt.suptitle('Automatic Ventricle Detection', fontsize=14)
plt.tight_layout()
plt.show()

<a id="medsam2-inference"></a>
## 5. Run MedSAM2 Inference

MedSAM2 treats the 3D volume as a "video" where each axial slice is a frame. The inference process:

1. **Resize** each grayscale slice to 512x512 RGB (MedSAM2's expected input size)
2. **Normalize** with ImageNet mean/std (the SAM2 backbone was pretrained on ImageNet)
3. **Initialize** the video predictor with all frames
4. **Prompt** with the bounding box on the key slice
5. **Propagate forward** through all superior slices (key_slice --> last slice)
6. **Reset** and re-prompt
7. **Propagate backward** through all inferior slices (key_slice --> first slice)
8. **Post-process** with largest connected component to remove spurious detections

Estimated inference time: ~30-60 seconds on T4 GPU for a 128-slice volume.

In [ ]:
# =============================================================================
# 5. Run MedSAM2 Inference
# =============================================================================
import torch
import time
from sam2.build_sam import build_sam2_video_predictor_npz
from skimage import measure

torch.set_float32_matmul_precision('high')


def resize_grayscale_to_rgb(array, image_size=512):
    """
    Resize (D, H, W) grayscale volume to (D, 3, image_size, image_size) RGB.

    MedSAM2 expects RGB input at 512x512 resolution.
    Each grayscale slice is replicated across 3 channels.
    """
    d, h, w = array.shape
    resized = np.zeros((d, 3, image_size, image_size), dtype=np.float32)
    for i in range(d):
        img_pil = Image.fromarray(array[i].astype(np.uint8))
        img_rgb = img_pil.convert("RGB")
        img_resized = img_rgb.resize((image_size, image_size), Image.BILINEAR)
        resized[i] = np.array(img_resized).transpose(2, 0, 1)
    return resized


def get_largest_cc(segmentation):
    """Keep only the largest connected component in a 3D binary mask."""
    labels = measure.label(segmentation, connectivity=2)
    if labels.max() == 0:
        return segmentation
    component_sizes = np.bincount(labels.flat)
    # Ignore background (label 0)
    component_sizes[0] = 0
    largest_label = np.argmax(component_sizes)
    return (labels == largest_label).astype(np.uint8)


# --- Initialize MedSAM2 predictor ---
checkpoint = './checkpoints/MedSAM2_2411.pt'
model_cfg = "configs/sam2.1_hiera_t512.yaml"

print("Loading MedSAM2 model...")
t0 = time.time()
predictor = build_sam2_video_predictor_npz(model_cfg, checkpoint)
print(f"Model loaded in {time.time() - t0:.1f}s")

# --- Preprocess volume for MedSAM2 ---
print("\nPreprocessing volume...")
video_height, video_width = img_3D.shape[1], img_3D.shape[2]
img_resized = resize_grayscale_to_rgb(img_3D, image_size=512)

# Normalize to [0, 1] then apply ImageNet normalization
img_resized = img_resized / 255.0
img_tensor = torch.from_numpy(img_resized).cuda().float()

img_mean = torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32)[:, None, None].cuda()
img_std = torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32)[:, None, None].cuda()
img_tensor -= img_mean
img_tensor /= img_std

print(f"Input tensor shape: {img_tensor.shape}")
print(f"Input tensor device: {img_tensor.device}")

# --- Scale bounding box to 512x512 coordinate space ---
scale_x = 512.0 / video_width
scale_y = 512.0 / video_height
bbox_scaled = np.array([
    ventricle_bbox[0] * scale_x,
    ventricle_bbox[1] * scale_y,
    ventricle_bbox[2] * scale_x,
    ventricle_bbox[3] * scale_y,
])
print(f"\nOriginal bbox: {ventricle_bbox}")
print(f"Scaled bbox (512x512): {bbox_scaled.astype(int)}")

# --- Run inference ---
segs_3D = np.zeros(img_3D.shape, dtype=np.uint8)

print(f"\nRunning MedSAM2 inference...")
print(f"  Key slice: {key_slice_idx}")
print(f"  Total slices: {img_3D.shape[0]}")
t_start = time.time()

with torch.inference_mode(), torch.autocast("cuda", dtype=torch.bfloat16):
    # --- Forward propagation (key_slice --> last slice) ---
    print("  Forward propagation...")
    inference_state = predictor.init_state(img_tensor, video_height, video_width)
    _, out_obj_ids, out_mask_logits = predictor.add_new_points_or_box(
        inference_state=inference_state,
        frame_idx=key_slice_idx,
        obj_id=1,
        box=bbox_scaled,
    )
    for out_frame_idx, out_obj_ids_f, out_mask_logits_f in predictor.propagate_in_video(inference_state):
        mask = (out_mask_logits_f[0] > 0.0).cpu().numpy()[0]
        segs_3D[out_frame_idx][mask] = 1
    predictor.reset_state(inference_state)

    # --- Backward propagation (key_slice --> first slice) ---
    print("  Backward propagation...")
    inference_state = predictor.init_state(img_tensor, video_height, video_width)
    _, out_obj_ids, out_mask_logits = predictor.add_new_points_or_box(
        inference_state=inference_state,
        frame_idx=key_slice_idx,
        obj_id=1,
        box=bbox_scaled,
    )
    for out_frame_idx, out_obj_ids_b, out_mask_logits_b in predictor.propagate_in_video(
        inference_state, reverse=True
    ):
        mask = (out_mask_logits_b[0] > 0.0).cpu().numpy()[0]
        segs_3D[out_frame_idx][mask] = 1
    predictor.reset_state(inference_state)

t_inference = time.time() - t_start
print(f"\nInference complete in {t_inference:.1f}s")
print(f"  Raw ventricle voxels: {np.sum(segs_3D)}")

# --- Post-processing: largest connected component ---
if np.max(segs_3D) > 0:
    segs_before = np.sum(segs_3D)
    segs_3D = get_largest_cc(segs_3D)
    segs_after = np.sum(segs_3D)
    removed = segs_before - segs_after
    print(f"  After largest CC: {segs_after} voxels ({removed} removed)")
else:
    print("  WARNING: No ventricle voxels detected!")

# Summary
slices_with_mask = np.sum(np.any(segs_3D, axis=(1, 2)))
print(f"\nSegmentation summary:")
print(f"  Mask shape: {segs_3D.shape}")
print(f"  Ventricle voxels: {np.sum(segs_3D)}")
print(f"  Slices with ventricle: {slices_with_mask}/{segs_3D.shape[0]}")
print(f"  Inference time: {t_inference:.1f}s")

<a id="visualize-results"></a>
## 6. Visualize Results

Display representative axial slices with the MedSAM2 segmentation overlay. Yellow highlights indicate ventricle regions detected by the model.

In [ ]:
# =============================================================================
# 6. Visualize Results
# =============================================================================

def show_mask(mask, ax, color=None, alpha=0.45):
    """Overlay a semi-transparent mask on a matplotlib axis."""
    if color is None:
        color = np.array([251 / 255, 252 / 255, 30 / 255])
    h, w = mask.shape[-2:]
    mask_rgba = np.zeros((h, w, 4))
    for c in range(3):
        mask_rgba[:, :, c] = color[c]
    mask_rgba[:, :, 3] = mask.astype(float) * alpha
    ax.imshow(mask_rgba)


# --- Panel 1: Representative axial slices ---
ventricle_slices = np.where(segs_3D.sum(axis=(1, 2)) > 0)[0]

if len(ventricle_slices) > 0:
    n_show = min(6, len(ventricle_slices))
    indices = np.linspace(0, len(ventricle_slices) - 1, n_show, dtype=int)
    selected = ventricle_slices[indices]

    fig, axes = plt.subplots(2, n_show, figsize=(4 * n_show, 8))
    if n_show == 1:
        axes = axes.reshape(2, 1)

    for i, z in enumerate(selected):
        # Top row: original brain-windowed slice
        axes[0, i].imshow(img_3D[z], cmap='gray')
        axes[0, i].set_title(f'Slice {z}', fontsize=11)
        axes[0, i].axis('off')

        # Bottom row: with MedSAM2 segmentation overlay
        axes[1, i].imshow(img_3D[z], cmap='gray')
        show_mask(segs_3D[z], axes[1, i])
        n_voxels = segs_3D[z].sum()
        axes[1, i].set_title(f'Seg ({n_voxels} px)', fontsize=11)
        axes[1, i].axis('off')

    plt.suptitle('MedSAM2 Ventricle Segmentation -- Axial Slices', fontsize=16, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("No ventricle voxels segmented. Check the bounding box prompt.")

# --- Panel 2: 3D coverage plot ---
fig, ax = plt.subplots(figsize=(10, 3))
slice_areas = segs_3D.sum(axis=(1, 2))
ax.fill_between(range(len(slice_areas)), slice_areas, alpha=0.6, color='gold')
ax.plot(slice_areas, color='orange', linewidth=1.5)
ax.axvline(key_slice_idx, color='cyan', linestyle='--', linewidth=1.5, label=f'Key slice ({key_slice_idx})')
ax.set_xlabel('Axial Slice Index')
ax.set_ylabel('Ventricle Area (pixels)')
ax.set_title('Ventricle Area Distribution Across Slices')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# --- Panel 3: Coronal and sagittal views ---
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# Coronal (mid Y)
mid_y = img_3D.shape[1] // 2
coronal_ct = img_3D[:, mid_y, :].T
coronal_mask = segs_3D[:, mid_y, :].T
axes[0].imshow(coronal_ct, cmap='gray', aspect='auto')
show_mask(coronal_mask, axes[0], alpha=0.5)
axes[0].set_title(f'Coronal View (y={mid_y})')
axes[0].set_xlabel('Slice (Z)')
axes[0].set_ylabel('X')

# Sagittal (mid X)
mid_x = img_3D.shape[2] // 2
sagittal_ct = img_3D[:, :, mid_x].T
sagittal_mask = segs_3D[:, :, mid_x].T
axes[1].imshow(sagittal_ct, cmap='gray', aspect='auto')
show_mask(sagittal_mask, axes[1], alpha=0.5)
axes[1].set_title(f'Sagittal View (x={mid_x})')
axes[1].set_xlabel('Slice (Z)')
axes[1].set_ylabel('Y')

plt.suptitle('MedSAM2 Ventricle Segmentation -- Orthogonal Views', fontsize=14)
plt.tight_layout()
plt.show()

<a id="morphometrics"></a>
## 7. Compute Morphometric Indices

Using the MedSAM2 ventricle segmentation, we compute the same morphometric indices as the HydroMorph RN app:

| Metric | Method | Abnormal Threshold |
|--------|--------|--------------------|
| **Evans Index** | Max frontal horn width / max inner skull width | > 0.3 |
| **Ventricle Volume** | Sum of segmented voxels x voxel volume | > 50 mL |
| **NPH Score** | Count of abnormal metrics | 0-2 (simplified) |

The **Evans Index** is computed by finding the axial slice where the ratio of ventricle width to inner skull width is maximized. The inner skull boundary is detected by thresholding for bone (HU > 300), filling holes, and eroding to find the inner table.

Note: Callosal Angle requires coronal slice identification at the posterior commissure level, which is not included in this simplified pipeline. See the full HydroMorph RN app for the complete 3-metric NPH score.

In [ ]:
# =============================================================================
# 7. Compute Morphometric Indices
# =============================================================================
from scipy.ndimage import binary_fill_holes, binary_erosion


def compute_evans_index(ct_volume_hu, vent_mask, spacing, bone_threshold=300):
    """
    Compute Evans Index: max frontal horn width / max inner skull width.

    Parameters
    ----------
    ct_volume_hu : np.ndarray
        (Z, Y, X) volume in original Hounsfield units.
    vent_mask : np.ndarray
        (Z, Y, X) binary ventricle mask.
    spacing : tuple
        Voxel spacing (sx, sy, sz) in mm.
    bone_threshold : float
        HU threshold for bone detection (default: 300).

    Returns
    -------
    best_evans : float
        Maximum Evans Index across all slices.
    best_slice : int
        Slice index where the maximum Evans Index was found.
    details : dict
        Detailed measurements at the best slice.
    """
    best_evans = 0.0
    best_slice = 0
    best_details = {}

    for z in range(vent_mask.shape[0]):
        if not np.any(vent_mask[z]):
            continue

        # Ventricle width on this slice (in mm)
        cols_with_vent = np.where(np.any(vent_mask[z], axis=0))[0]
        if len(cols_with_vent) < 2:
            continue
        vent_width_mm = float(cols_with_vent[-1] - cols_with_vent[0]) * spacing[0]

        # Inner skull width (in mm)
        bone_mask = ct_volume_hu[z] > bone_threshold
        if not np.any(bone_mask):
            continue
        filled = binary_fill_holes(bone_mask)
        struct = np.ones((3, 3))
        inner = binary_erosion(filled, structure=struct, iterations=2)
        if not np.any(inner):
            inner = filled

        cols_skull = np.where(np.any(inner, axis=0))[0]
        if len(cols_skull) < 2:
            continue
        skull_width_mm = float(cols_skull[-1] - cols_skull[0]) * spacing[0]

        # Sanity check: skull should be at least 50mm wide
        if skull_width_mm < 50.0:
            continue

        evans = vent_width_mm / skull_width_mm
        if evans > best_evans:
            best_evans = evans
            best_slice = z
            best_details = {
                'ventricle_width_mm': vent_width_mm,
                'skull_width_mm': skull_width_mm,
                'ventricle_cols': (int(cols_with_vent[0]), int(cols_with_vent[-1])),
                'skull_cols': (int(cols_skull[0]), int(cols_skull[-1])),
            }

    return best_evans, best_slice, best_details


def compute_ventricle_volume(vent_mask, spacing):
    """
    Compute ventricle volume in milliliters.

    Parameters
    ----------
    vent_mask : np.ndarray
        Binary ventricle mask.
    spacing : tuple
        Voxel spacing in mm.

    Returns
    -------
    volume_ml : float
        Ventricle volume in mL (= cm^3).
    """
    voxel_vol_mm3 = float(np.prod(spacing[:3]))
    n_voxels = float(np.sum(vent_mask > 0))
    return (n_voxels * voxel_vol_mm3) / 1000.0


# Load original HU for Evans Index computation
nii_orig = nib.load(nifti_path)
vol_hu_orig = nii_orig.get_fdata().astype(np.float32)

# Reorient and transpose to match MedSAM2 output format
ornt = nib.orientations.io_orientation(nii_orig.affine)
ras_ornt = nib.orientations.axcodes2ornt(('R', 'A', 'S'))
if not np.array_equal(ornt, ras_ornt):
    transform = nib.orientations.ornt_transform(ornt, ras_ornt)
    vol_hu_orig = nib.orientations.apply_orientation(vol_hu_orig, transform)
vol_hu_zyx = np.transpose(vol_hu_orig, (2, 1, 0))

# --- Compute metrics ---
evans_idx, evans_slice, evans_details = compute_evans_index(vol_hu_zyx, segs_3D, spacing)
vent_vol_ml = compute_ventricle_volume(segs_3D, spacing)

# --- Display results ---
print("=" * 60)
print("  MORPHOMETRIC RESULTS (MedSAM2 Segmentation)")
print("=" * 60)
print()
print(f"  Evans Index:        {evans_idx:.4f}", end="")
print(f"    {'[ABNORMAL > 0.3]' if evans_idx > 0.3 else '[normal]'}")
if evans_details:
    print(f"    Ventricle width:  {evans_details['ventricle_width_mm']:.1f} mm")
    print(f"    Skull width:      {evans_details['skull_width_mm']:.1f} mm")
    print(f"    Measured at:      slice {evans_slice}")
print()
print(f"  Ventricle Volume:   {vent_vol_ml:.1f} mL", end="")
print(f"    {'[ABNORMAL > 50 mL]' if vent_vol_ml > 50 else '[normal]'}")
print(f"    Total voxels:     {np.sum(segs_3D)}")
print(f"    Voxel volume:     {np.prod(spacing[:3]):.2f} mm^3")
print()

# NPH Score (simplified: 2 metrics)
nph_score = 0
if evans_idx > 0.3:
    nph_score += 1
if vent_vol_ml > 50:
    nph_score += 1

risk_label = {0: 'LOW', 1: 'MODERATE', 2: 'HIGH'}[nph_score]
print(f"  NPH Score:          {nph_score}/2")
print(f"  NPH Risk:           {risk_label}")
print()
print("=" * 60)
print("  Note: Full NPH score (0-3) includes Callosal Angle,")
print("  which requires the complete HydroMorph RN pipeline.")
print("=" * 60)

# --- Visualize Evans Index measurement ---
if evans_details:
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(img_3D[evans_slice], cmap='gray')
    show_mask(segs_3D[evans_slice], ax, alpha=0.3)

    # Draw ventricle width line
    vc = evans_details['ventricle_cols']
    mid_row = img_3D.shape[1] // 2
    ax.plot([vc[0], vc[1]], [mid_row, mid_row], 'c-', linewidth=2,
            label=f'Ventricle: {evans_details["ventricle_width_mm"]:.1f} mm')

    # Draw skull width line
    sc = evans_details['skull_cols']
    ax.plot([sc[0], sc[1]], [mid_row + 10, mid_row + 10], 'r-', linewidth=2,
            label=f'Skull: {evans_details["skull_width_mm"]:.1f} mm')

    ax.set_title(f'Evans Index = {evans_idx:.4f} (slice {evans_slice})', fontsize=14)
    ax.legend(loc='lower right', fontsize=11)
    ax.axis('off')
    plt.tight_layout()
    plt.show()

<a id="save-results"></a>
## 8. Save Results

Save the segmentation mask as a NIfTI file and export brain-windowed PNG slices with overlays. These outputs can be:
- Loaded in ITK-SNAP or 3D Slicer for validation
- Used as real sample images in the HydroMorph RN app
- Compared with the classical pipeline segmentation

In [ ]:
# =============================================================================
# 8. Save Results
# =============================================================================
import json
from datetime import datetime

# --- Save segmentation mask as NIfTI ---
# Transpose back from (Z, Y, X) to (X, Y, Z) for NIfTI convention
mask_xyz = np.transpose(segs_3D, (2, 1, 0))
mask_nii = nib.Nifti1Image(mask_xyz.astype(np.uint8), nii.affine)
output_mask_path = 'data/ventricle_mask_medsam2.nii.gz'
nib.save(mask_nii, output_mask_path)
print(f"Ventricle mask saved: {output_mask_path}")
print(f"  Shape: {mask_xyz.shape}")
print(f"  File size: {os.path.getsize(output_mask_path) / 1e3:.1f} KB")

# --- Save brain-windowed PNGs with overlays ---
png_dir = 'data/ventricle_slices_png'
os.makedirs(png_dir, exist_ok=True)

saved_count = 0
for z in range(img_3D.shape[0]):
    if segs_3D[z].sum() > 0:
        # Save brain-windowed slice
        slice_png = Image.fromarray(img_3D[z])
        slice_png.save(os.path.join(png_dir, f'slice_{z:04d}.png'))

        # Save overlay (yellow ventricle highlighting)
        rgb = np.stack([img_3D[z]] * 3, axis=-1).copy()
        mask_overlay = segs_3D[z].astype(bool)
        rgb[mask_overlay, 0] = 255  # R
        rgb[mask_overlay, 1] = 255  # G
        rgb[mask_overlay, 2] = 30   # B (yellow)
        overlay_png = Image.fromarray(rgb)
        overlay_png.save(os.path.join(png_dir, f'slice_{z:04d}_overlay.png'))
        saved_count += 1

print(f"\nPNG slices saved: {saved_count} slice pairs to {png_dir}/")

# --- Save morphometric results as JSON ---
results = {
    'timestamp': datetime.now().isoformat(),
    'model': 'MedSAM2',
    'checkpoint': 'MedSAM2_2411.pt',
    'input_file': nifti_path,
    'input_shape': list(volume.shape),
    'spacing_mm': list(spacing),
    'key_slice': int(key_slice_idx),
    'bounding_box': ventricle_bbox.tolist(),
    'metrics': {
        'evans_index': float(evans_idx),
        'evans_slice': int(evans_slice),
        'evans_abnormal': bool(evans_idx > 0.3),
        'ventricle_volume_ml': float(vent_vol_ml),
        'volume_abnormal': bool(vent_vol_ml > 50),
        'ventricle_voxels': int(np.sum(segs_3D)),
        'nph_score': nph_score,
        'nph_risk': risk_label,
    },
    'inference_time_s': float(t_inference),
}

if evans_details:
    results['metrics']['ventricle_width_mm'] = float(evans_details['ventricle_width_mm'])
    results['metrics']['skull_width_mm'] = float(evans_details['skull_width_mm'])

results_path = 'data/medsam2_results.json'
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f"\nResults JSON saved: {results_path}")

# --- Download results (Colab) ---
# Uncomment the following lines to download results to your local machine:
# from google.colab import files
# files.download(output_mask_path)
# files.download(results_path)

print(f"\nAll outputs saved to data/ directory:")
print(f"  {output_mask_path} -- NIfTI segmentation mask")
print(f"  {png_dir}/ -- Brain-windowed PNGs with overlays")
print(f"  {results_path} -- Morphometric results JSON")

<a id="next-steps"></a>
## 9. Next Steps

### Improving Segmentation Quality
- **Fine-tune MedSAM2** on ventricle-specific data with expert annotations (see `medsam2_ventricle_finetuning.ipynb`)
- **Multi-prompt strategy:** Use bounding boxes on multiple slices (e.g., key slice + 2 adjacent) for more robust propagation
- **Point prompts:** Add positive/negative point prompts in addition to the bounding box for difficult cases

### Deployment Options
- **HuggingFace Spaces:** Deploy fine-tuned model as a Gradio API, call from HydroMorph RN app via the existing Gradio API infrastructure in `src/models/ApiModelProvider.js`
- **Efficient MedSAM2 Tiny** (`eff_medsam2_tiny_FLARE25_RECIST_baseline.pt`, 71.7 MB): CPU-capable variant for lower-resource deployment
- **ONNX export:** Convert to ONNX for cross-platform inference without PyTorch dependency

### Integration with HydroMorph RN
- Replace `MockModelProvider.js` MedSAM2 mock with real API calls to the deployed model
- Pass NIfTI file to the Gradio endpoint, receive segmentation mask
- Compute morphometrics on-device using the returned mask (privacy-preserving: only mask is transmitted, not the original scan)

### Comparison & Validation
- Compare MedSAM2 results with MONAI models (see `monai_brain_segmentation.ipynb`)
- Validate against expert manual segmentations using Dice score, Hausdorff distance
- Cross-validate Evans Index and volume measurements against radiologist ground truth

### References
- [MedSAM2 GitHub](https://github.com/bowang-lab/MedSAM2)
- [MedSAM2 Paper](https://arxiv.org/abs/2408.00874)
- [SAM2 Paper (Meta)](https://arxiv.org/abs/2408.00714)
- [Evans Index - Radiopedia](https://radiopaedia.org/articles/evans-index)
- [HydroMorph RN App](https://github.com/yourusername/hydromorph-rn)